## F.log_softmax(shift_logits, dim=-1) 
- 这行代码把模型原始 logits 转换成对数概率
- 为后续计算负对数似然损失（NLLLoss）或手动实现交叉熵做准备

- <b>log_softmax 内部已做 log-sum-exp trick，比“先 softmax 再 log”更稳</b>

### “先减 max，再 log-sum-exp，最后减回去”——记住这三步，任何框架都能 5 行写完 log_softmax

#### 教学版：numpy

In [2]:
import numpy as np

def log_softmax(x,axis=-1):
    x_max = np.max(x,axis=axis,keepdims=True)
    print(x_max)
    # 减 max 防溢出
    log_sum_exp = np.log(np.sum(np.exp(x-x_max),axis=axis,keepdims=True))
    return x - x_max - log_sum_exp

# demo

a = np.array([[1.0,2.0,3.0],
             [1.0,2.0,5.0]])
print(log_softmax(a))

[[3.]
 [5.]]
[[-2.40760596 -1.40760596 -0.40760596]
 [-4.0658839  -3.0658839  -0.0658839 ]]


#### pytorch (含梯度)

In [3]:
import torch
import math

def log_softmax_torch(x,dim=-1):
    x_max = torch.amax(x,dim=dim,keepdim=True)
    log_sum_exp = torch.log(torch.sum(torch.exp(x-x_max),dim=dim,keepdim=True))
    return x - x_max - log_sum_exp

# 与官方对比
x = torch.randn(2,5,8,requires_grad=True)
loss1 = log_softmax_torch(x).sum()
loss2 = torch.log_softmax(x,dim=-1).sum()

print(torch.allclose(loss1,loss2,atol=1e-6))

True


#### 高效版： 利用torch.logsumexp


In [4]:
def log_softmax_fast(x,dim=-1):
    return x - torch.logsumexp(x,dim=dim,keepdim=True)

x = torch.randn(2,5,8,requires_grad=True)
loss = log_softmax_fast(x)

print(loss)

tensor([[[-1.2669, -2.0115, -2.2846, -2.9255, -2.3677, -2.6327, -1.7665,
          -2.3799],
         [-2.2613, -2.7066, -4.3152, -2.2835, -2.0056, -0.6777, -2.9144,
          -4.0671],
         [-0.9222, -4.8067, -3.9115, -2.5930, -1.6202, -2.9053, -3.5939,
          -1.5173],
         [-1.9759, -2.4879, -3.5463, -2.7346, -1.8227, -2.2116, -1.6289,
          -1.5266],
         [-1.9694, -3.1765, -1.5437, -1.0306, -2.4296, -2.5126, -3.4849,
          -3.0253]],

        [[-3.0134, -1.4786, -1.3904, -2.5540, -1.9919, -1.9195, -2.9289,
          -2.8200],
         [-2.7899, -1.2249, -2.7233, -1.0449, -3.4171, -2.7097, -2.8014,
          -2.6981],
         [-2.0313, -3.6997, -1.9641, -1.9186, -2.0740, -1.1552, -2.6748,
          -3.0485],
         [-3.0881, -3.3607, -2.9399, -0.5525, -2.0225, -3.8025, -2.5775,
          -2.8017],
         [-2.9861, -2.2286, -1.6984, -4.6209, -3.1988, -2.0847, -1.0224,
          -2.0864]]], grad_fn=<SubBackward0>)


In [1]:
import torch,torch.nn.functional as F

logits = torch.randn(2,10,50256)
labels = torch.randint(0,50256,(2,10))

# 步骤1：手动负对数似然
log_probs = F.log_softmax(logits,dim=-1)
loss = F.nll_loss(log_probs.view(-1,50256),labels.view(-1),ignore_index=-100)

# 与步骤1等价
loss2 = F.cross_entropy(logits.view(-1,50256),labels.view(-1),ignore_index=-100)

# 数值相等
print(loss.item(),loss2.item())

11.04072380065918 11.04072380065918


## F.nll_loss
- PyTorch 的 负对数似然损失
- 专为已归一化的对数概率设计

常用于：
- 手写交叉熵（配合log_softmax)
- 标签平滑、蒸馏、强化学习等需要’修改概率‘的场景

In [ ]:
torch.nn.functional.nll_loss(
    input,               # 对数概率  [N, C] 或 [N, C, d1, d2, ...]
    target,              # 类别索引  [N]    或 [N, d1, d2, ...]
    weight=None,         # 各类别权重  [C]
    ignore_index=-100,   # 要屏蔽的目标值
    reduction='mean'     # 'none' | 'mean' | 'sum'
)

### 常用参数选项
#### ignore_index 
- 把target中等于ignore_index的位置从loss里剔除(HF默认-100)


In [ ]:
loss = F.nll_loss(log_probs,labels,ignore_index=-100)

### weight
- 解决类别不平衡

In [ ]:
weight = torch.tensor([1.0,2.0,5.0])
loss = F.nll_loss(log_probs,labels,weight)

### reduction = 'none'
- 不做规约，返回逐元素的loss，方便后续手动加权或做RL

In [ ]:
loss = F.nll_loss(log_probs,labels,reduction='none')

## nll_loss 使用

In [ ]:
# 输入必须是log_probs

log_probs = F.log_softmax(logits,dim=1)
loss = F.nll_loss(log_probs,labels)

# 千万别把原始 logits 直接喂给 nll_loss，否则数值意义完全错误（它会当成 log-prob 解释， loss 可能为负）。

## 使用nll_loss的完整例子


In [1]:
import torch,torch.nn.functional as F

logits = torch.randn(2,10,50256) # [B,L,V]
# 在区间 [0, 50256) 内均匀随机采样整数，返回一个形状为 (2, 10) 的 LongTenso
labels = torch.randint(0,50256,(2,10)) # [B,L]
print(logits)
print(labels)

# 1 左移对齐
shift_logits = logits[...,:-1,:] # [B,L-1,V]
shift_labels = labels[...,1:]    # [B,L-1]

# 2 转2D
log_probs = F.log_softmax(shift_logits,dim=-1).view(-1,50256) # [B *(L-1),V]
flat_labels = shift_labels.reshape(-1) # [B * (L-1)]

# 3 计算NLL
loss = F.nll_loss(log_probs,flat_labels,ignore_index=-100)
print(loss.item())

tensor([[[-0.0873,  1.4973,  0.8404,  ...,  0.0717, -1.5595,  0.0445],
         [-0.4903,  0.8457, -2.1043,  ...,  0.7861,  0.4364,  0.0853],
         [-0.5923, -0.0667, -1.3581,  ...,  0.9232, -1.5429, -0.3139],
         ...,
         [ 0.1583, -1.8290,  2.3305,  ...,  0.0079,  0.9055, -0.4204],
         [-0.7862, -0.8623,  1.6171,  ..., -2.4676,  0.7615,  0.5908],
         [-1.2959, -0.0554,  0.4789,  ...,  0.3327,  1.4617, -0.3399]],

        [[-1.1563,  0.8954, -1.8313,  ...,  0.5105,  1.8464,  0.4109],
         [-0.7437, -0.5920, -1.6624,  ...,  0.6418,  1.4834,  0.8156],
         [-0.1386, -0.3327,  0.1776,  ..., -1.8581, -0.2611,  0.3344],
         ...,
         [ 0.4700, -1.4395,  0.3947,  ...,  0.7644,  0.7177,  0.2306],
         [-0.2718, -1.1411,  0.6726,  ..., -1.0890,  1.3492, -0.5584],
         [ 0.6201,  1.7958, -0.7243,  ...,  1.4600, -0.3027, -0.8608]]])
tensor([[24054, 31089, 25966,  8376, 16596,  1612, 44543, 31052, 32973, 15350],
        [25045,  4668, 13118, 30801,

## 源码实现F.nll_loss

只支持最常用的二维输入 [N, C]（可扩展到多维，思路相同）。
核心步骤：
- 取出每个样本对应类别的 log-probability
- 根据 weight 重新加权
- 按 ignore_index 打掩码
- 做 reduction

In [13]:
import torch
import torch.nn.functional as F

def nll_loss_py(log_prob1,target,weight=None,ignore_index=-100,reduction='mean'):
    """
    纯 Python / PyTorch 实现 F.nll_loss 的核心逻辑。
    log_prob : [N, C]  已经过 log_softmax
    target   : [N]     类别索引 0..C-1
    weight   : [C]     各类别权重（可选）
    """
    N,C = log_prob1.shape
    print(N,C)
    
    # 1 构造行索引
    rng = torch.arange(N,device=log_prob1.device)
    
   
    
    # 2 忽略ignore_index
    # 使用mask，将=ignore_index的位设置为false
    mask = target != ignore_index
    # 对于 ignore_index 位置，临时把 target 换成 0 避免越界，后面再掩码
    safe_target = target.clone()
    
    print('safe_target:',safe_target)
    safe_target[~mask] = 0
    print('mask:',mask)
    print('safe_target_with_mask:',safe_target)
    
    print('log_prob1:',log_prob1)
    
    print('rng:',rng)
    print('target:',safe_target)
    # 3 取出对应类别的 -logP(注意nll_loss 公式里自带负号)
    loss = -log_prob1[rng,safe_target] # [N]
    
    print('loss:',loss)
    print('mask:',mask)
    loss = loss * mask
    print('loss_with_mask:',loss)
        
    # 4 应用类别权重
    if weight is not None:
        loss = loss * weight[target]
        
   
    # 5 reduction 
    if reduction == 'none':
        return loss
    elif reduction == 'sum':
        return loss.sum()
    elif reduction =='mean':
        return loss.sum()/mask.sum() # 只平均有小位置
    else:
        raise ValueError("reduction must be 'none'|'mean'|'sum'")

# 单元测试
logits = torch.randn(5,8,requires_grad=True)
log_prob = F.log_softmax(logits,dim=1)
target = torch.tensor([3,0,7,1,-100]) # 最后一个是ignore

loss_ref = F.nll_loss(log_prob,target,ignore_index=-100)
loss_my = nll_loss_py(log_prob,target,ignore_index=-100)

print('diff:',abs(loss_ref - loss_my).item()) # 0.0

loss_my.backward()

5 8
safe_target: tensor([   3,    0,    7,    1, -100])
mask: tensor([ True,  True,  True,  True, False])
safe_target_with_mask: tensor([3, 0, 7, 1, 0])
log_prob1: tensor([[-0.6601, -1.9121, -3.9594, -2.1001, -3.3739, -3.4943, -2.2966, -3.5514],
        [-3.8783, -2.4920, -2.0404, -1.6316, -1.4573, -2.2612, -1.8630, -2.5424],
        [-1.7637, -0.9040, -4.7066, -4.4373, -3.3450, -2.6456, -3.4801, -1.3252],
        [-3.1326, -2.4561, -5.3655, -3.1352, -4.1946, -1.6966, -0.6709, -2.1821],
        [-2.2329, -1.8506, -1.9943, -1.5047, -3.7690, -1.5643, -3.3437, -2.2088]],
       grad_fn=<LogSoftmaxBackward0>)
rng: tensor([0, 1, 2, 3, 4])
target: tensor([3, 0, 7, 1, 0])
loss: tensor([2.1001, 3.8783, 1.3252, 2.4561, 2.2329], grad_fn=<NegBackward0>)
mask: tensor([ True,  True,  True,  True, False])
loss_with_mask: tensor([2.1001, 3.8783, 1.3252, 2.4561, 0.0000], grad_fn=<MulBackward0>)
diff: 0.0


与官方 C++ 版本差异
- 未做多维 ([N,C,d1,d2,...]) 的自动展开；
- 未对 weight 做 dtype/device 检查；
- 未集成 kernel fuse，速度略慢。

但数值、梯度、掩码、加权、reduction 行为均一致，可当作“白盒”参考。